**导入包、定义函数**

In [ ]:
from traversal_engine import TraversalEngine
from function_registry import FUNCTION_REGISTRY
import traversal_functions  # 导入，注册函数
from neo4j import GraphDatabase
import veripg_construct
import copy
import graph_pattern as gp
import time
import os
from pyverilog.vparser.parser import parse
import glob
import json
import csv
import pandas as pd
from tqdm.notebook import tqdm

class DetectionResult:
    # DetectionResult class to store results
    def __init__(self, file_name, cwe, result, exec_time, exec_counter, nodes_count, relationships_count):
        self.file_name = file_name
        self.cwe = cwe
        self.result = result
        self.exec_time = exec_time
        self.exec_counter = exec_counter
        self.nodes_count = nodes_count
        self.relationships_count = relationships_count

def detect_one(file_name, query):
    print(f"detecting file: {file_name}")
    with GraphDatabase.driver(uri, auth=(username, password)) as driver:
        nodes_count, relationships_count = veripg_construct.VeriPG(file_name, driver)
        with driver.session() as session:
            engine = TraversalEngine(session, FUNCTION_REGISTRY)
            
            start_time = time.time()
            result = engine.execute(query)
            end_time = time.time()

    return DetectionResult(file_name, query, result, end_time - start_time, engine.exec_counter, nodes_count, relationships_count)

def detect_all(file_name, query_dict):
    print(f"detecting file: {file_name}")
    with GraphDatabase.driver(uri, auth=(username, password)) as driver:
        nodes_count, relationships_count = veripg_construct.VeriPG(file_name, driver)
        with driver.session() as session:
            engine = TraversalEngine(session, FUNCTION_REGISTRY)

            detect_results = []
            # i=0
            counter=0
            for cwe, query in query_dict.items():
                # print(f'{i}: -----------------------------------------------------------')
                # i+=1
                start_time = time.time()
                result = engine.execute(query)
                end_time = time.time()
                counter=engine.exec_counter-counter
                detect_results.append(DetectionResult(file_name, cwe, result, end_time - start_time, counter, nodes_count, relationships_count))
                counter=engine.exec_counter

    return detect_results

def construct(file_name):
    print(f"constructing file: {file_name}")
    driver = GraphDatabase.driver(uri, auth=(username, password))
    veripg_construct.VeriPG(file_name, driver)
    driver.close()

**漏洞检测：** 对所有Verilog代码进行全面检测

In [2]:
construct_files_path_handle = f"../dataset_collection/dataset/construct_path_handle.csv"
detect_results_file_path = f"../dataset_collection/dataset/detect_results_realworld_cwe_1271.csv"
df=pd.read_csv(construct_files_path_handle)
query_list={'1271': gp.all_gp_dict.get('1271')}
detect_results_existing = pd.read_csv(detect_results_file_path)
existing_files = detect_results_existing['file'].unique().tolist()
f = open(detect_results_file_path, 'a+', newline='')
writer = csv.writer(f)
pbar = tqdm(total=len(df), desc="Detecting Vulnerabilities in Verilog Files")
for index, data in df.iterrows():
    file_path=f"../dataset_collection/"+data['file_path']
    if file_path in existing_files:
        pbar.update(1)
        continue

    drs = detect_all(file_path, query_list)
    for dr in drs:
        writer.writerow([ data['repo'], dr.file_name, dr.cwe, dr.result, dr.exec_time, dr.exec_counter, dr.nodes_count, dr.relationships_count])
        print(f"File: {data['file_path']}, Results: {dr.result}, Time: {dr.exec_time}, Exec Count: {dr.exec_counter}, Nodes Count: {dr.nodes_count}, Relationships Count: {dr.relationships_count} \n")
    # f.flush()
    pbar.update(1)
f.close()

Detecting Vulnerabilities in Verilog Files:   0%|          | 0/785 [00:00<?, ?it/s]

**漏洞检测：** 单文件单漏洞检测

In [3]:
file_name = f"../dataset_collection/dataset/filestructure_hackatdac18/ips/hwpe-mac-engine/rtl/md5.v"
query = gp.all_gp_dict['1271']
detect_one(file_name, query).result

detecting file: ../dataset_collection/dataset/filestructure_hackatdac18/ips/hwpe-mac-engine/rtl/md5.v


Generating LALR tables


delete_all_nodes
ast2neo4j
set_assign_relationship
delete_node(Source)
cfg2neo4j
ddg2neo4j
value_map_id: {'clk': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20088', 'reset': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20090', 'load_i': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20092', 'ready_o': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20108', 'newtext_i': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20096', 'data_i': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20098', 'data_o': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20111', 'round64': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20120', 't': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20129', 'ar': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20134', 'A': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20171', 'message': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20204', 'round': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20213', 'next_generate_hash': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20222', 'hash_generated': '4:d391ffb5-0d39-4ef2-b589-d7d42215ef3d:20225', 'next_getdata_sta

[]

**数据处理**：读取的通过sv2v转换的Verilog代码

In [ ]:
verilog_path_files = f"../dataset_collection/dataset/verilog_path.csv"
VerilogPathDF = pd.read_csv(verilog_path_files).dropna(subset=['new_file'])
VerilogPathDF

**数据处理**：尝试将Verilog代码构造成VeriPG，并保留可构造的文件

In [ ]:
construct_files_path = "../dataset_collection/dataset/construct_path.csv"
ConstructPathList = pd.read_csv(construct_files_path)['file_path'].tolist()
f=open(construct_files_path, 'a+', newline='')
writer = csv.writer(f)
# writer.writerow(['repo','file_path','error'])
pbar = tqdm(total=len(VerilogPathDF), desc="Constructing Verilog Files")
for index, row in VerilogPathDF.iterrows():
    if row['new_file'] in ConstructPathList:
        pbar.update()
        continue
    file_path = f"../dataset_collection/{row['new_file']}"
    try:
        construct(file_path)
        writer.writerow([row['repo'], row['new_file'], ''])
    except Exception as e:
        writer.writerow([row['repo'], row['new_file'], str(e)])
    pbar.update()
f.close()

# 数据清洗：去除不能构造VeriPG的代码
construct_files_path = "../dataset_collection/dataset/construct_path.csv"
construct_files_path_handle = "../dataset_collection/dataset/construct_path_handle.csv"
df=pd.read_csv(construct_files_path)
df.drop_duplicates(subset=['file_path'], keep='first', inplace=True)
df = df[df['error'].isna()]
df.to_csv(construct_files_path_handle, index=False)

废弃的检测代码

In [ ]:
# v_files = glob.glob(os.path.join("..\data","**","*.v"), recursive=True)

# if __name__ == "__main__":

#     line_count_list = []

#     for v_file in v_files:
#         file_name = os.path.basename(v_file)  # 获取文件名
#         try:
#             line_count = sum(1 for _ in open(v_file, "r", encoding="utf-8"))  # 计算行数
#         except Exception as e:
#             print(f"Error reading {v_file}: {e}")
#             line_count = 0
#         line_count_list.append(line_count)

#     csv_output_file = "..\data\experiment_result\scanner_time.csv"
#     csv_output_file2 = "..\data\experiment_result\scanner_execcount.csv"
#     with open(csv_output_file, mode="a+", newline="", encoding="utf-8") as csvfile:
#         writer = csv.writer(csvfile)
#         writer.writerow(["File Name"]+[os.path.basename(v_file) for v_file in v_files])  # 将 v_files 的内容作为第一行写入
#         writer.writerow(["Line Count"] + line_count_list)  # 将行数写入第二行
#     with open(csv_output_file2, mode="a+", newline="", encoding="utf-8") as csvfile:
#         writer = csv.writer(csvfile)
#         writer.writerow(["File Name"]+[os.path.basename(v_file) for v_file in v_files])  # 将 v_files 的内容作为第一行写入
#         writer.writerow(["Line Count"] + line_count_list)  # 将行数写入第二行
#     # batch = True
#     # # batch = False
#     # if batch:
#     #     # cweid="226"
#     #     # query = query1245_ds
#     #     query_list = [query226_ds,query1231_ds,query1232_ds,query1244_ds,query1255_ds,query1300_ds,query1258_gpt,query1271_gpt_3,query1280_ds]
#     #     # query_name = "1245ds"
#     #     query_name_list = ["226ds","1231ds","1232ds","1244ds","1255ds","1300ds","1258gpt","1271gpt","1280ds"]
#     #     # result_file = "..\data\experiment_result\cwe_vul_gpt4o_our.json"
#     #     # result_file = "..\data\experiment_result\cwe_vul_deepseek_our.json"
#     #     result_file = "..\data\experiment_result\scanner_time.csv"
#     #     result_file2 = "..\data\experiment_result\scanner_execcount.csv"

#     #     print(f"Found {len(v_files)} .v files in ..\data.")
#     #     for v_file in v_files:
#     #         print(v_file)
        
#     #     for i,query in enumerate(query_list):
#     #         query_name = query_name_list[i]
#     #         # results = []
#     #         run_time_list = []
#     #         exec_counter_list = []
#     #         for v_file in v_files:
#     #             run_time, counter = detect(v_file, copy.deepcopy(query))
#     #             run_time_list.append(run_time)
#     #             exec_counter_list.append(counter)

#     #             # print("result:")
#     #             # print(result)
#     #             # if result is not None:
#     #             #     results.append(v_file)
#     #             # if result:
#     #             #     results.append(v_file)
#     #         with open(result_file, "a+", newline="", encoding="utf-8") as csvfile:
#     #             writer = csv.writer(csvfile)
#     #             writer.writerow([query_name]+run_time_list)
#     #         with open(result_file2, "a+", newline="", encoding="utf-8") as csvfile:
#     #             writer = csv.writer(csvfile)
#     #             writer.writerow([query_name]+exec_counter_list)
#     #         # print("results:")
#     #         # for res in results:
#     #         #     print(res)
#     #         # with open(result_file, "a+") as f:
#     #         #     output = {cweid: results}
#     #         #     json.dump(output, f, indent=4, ensure_ascii=False)
#     # else:
#     #     # print(detect("..\\data\\ImproperLock\\1234\\1234_3.v", query1280_ds))
#     #     # print(detect("..\\data\\ImproperAccessControl\\1280\\1280_1.v", query1280_ds))
#     #     # print(detect("..\\data\\ImproperResourceOperate\\1271\\1271_3.v", query1271_gpt_2))
#     #     # print(detect("..\\data\\ImproperResourceOperate\\1258\\1258_4.v", query1258_gpt))
#     #     # print(detect("..\\data\\IncorrectAuthorization\\1244\\1244_1.v", query1244_ds))
#     #     print(detect("..\\data\\SideChannel\\1255\\1255_4.v", query1243_ds))
#     #     # print(detect("..\\data\\SideChannel\\1300\\1300_4.v", query1300_gpt))
#     #     # print(detect("..\\data\\FinidetectateMachine\\1245\\1245_1.v", query1243_ds))
#     #     # print(detect("..\\data\\FinidetectateMachine\\1245\\1245_1.v", query1271_gpt_2))
#     #     # print(detect("..\\data\\ImproperLock\\1232\\1232_4.v", query1232_ds))
#     #     # print(detect("..\\data\\NonVulnerability\\1.v", query1245_gpt))
#     #     # detect("..\\data\\ImproperResourceOperate\\226\\226_4.v", query226_2)
#     #     # print(query226_2)